<div dir="rtl">
<h1>از جدول مقایسه تا بردار پاسخ</h1>
<p>درس 37 از 76 · از وزن‌ها تا خروجی کامل Attention · <code dir="ltr">31-values</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-02/31-values.html">📖 بازگشت به همین درس</a></p>
<p>Softmax سطری و ترکیب Valueها را در یک تابع قابل آزمون به هم وصل کنید.</p><p>پیش‌نیاز: امتیاز مقیاس‌شده و جمع وزن‌دار را از دو درس قبل بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>دو Query با امتیازهای متفاوت داریم. اگر فقط Valueها دو برابر شوند، کدام خروجی تابع باید ثابت بماند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
scores = torch.tensor([[1.,0.,-1.],[0.,0.,0.]])
values = torch.tensor([[10.,0.],[0.,20.],[-10.,5.]])
print('scores:',scores,'values:',values)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع attend(scores, values) یک زوج (weights, output) برگرداند. Softmax را بدون torch.softmax بسازید: بیشینهٔ هر سطر را کم کنید، exp بگیرید و بر مجموع همان سطر تقسیم کنید؛ سپس Valueها را ترکیب کنید.</p>
</div>

In [ ]:
def attend(scores, values):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = attend(scores,values)
    if result is None: return False
    weights,out = result
    torch.testing.assert_close(weights,scores.softmax(-1))
    torch.testing.assert_close(out,scores.softmax(-1)@values)
    torch.testing.assert_close(out[1],values.mean(0))
    large = torch.tensor([[1000.,1001.,999.]])
    w,y = attend(large,values)
    assert torch.isfinite(w).all() and torch.isfinite(y).all()
    torch.testing.assert_close(w,large.softmax(-1))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>برای جداکردن اثر Value، در این آزمایش وزن‌ها را از قبل تعیین کرده‌ایم؛ این جدول از scores محاسبه نشده است. فقط ضریب Value را از یک به دو تغییر دهید و خروجی را مقایسه کنید، نه فقط Shape را.</p>
</div>

In [ ]:
fixed_weights = torch.tensor([[0.7,0.2,0.1],[1/3,1/3,1/3]])
print('original:',fixed_weights@values)
print('doubled values:',fixed_weights@(2*values))
print('same weights:',fixed_weights)

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>کد خراب روی محور Query نرمال می‌کند. تابع key_weights(scores) را اصلاح کنید تا هر Query توزیع خودش را روی Keyها داشته باشد.</p>
</div>

In [ ]:
wrong = scores.softmax(0)
print('wrong row sums:',wrong.sum(-1),'column sums:',wrong.sum(0))

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def key_weights(scores):
    # TODO
    return None

In [ ]:
def test_repair():
    result = key_weights(scores)
    if result is None: return False
    torch.testing.assert_close(result.sum(-1),torch.ones(2))
    torch.testing.assert_close(result,scores.softmax(-1))
    other = torch.tensor([[1.,2.],[0.,0.],[-2.,3.]])
    torch.testing.assert_close(key_weights(other),other.softmax(-1))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>دو خروجی تابع با weights و weighted_values در trace Attention پروژه متناظرند؛ در این تمرین Dropout نداریم. محاسبه هنوز بدون Mask است و برای آموزش علّی کافی نیست.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر وزن‌ها مجموع یک داشته باشند اما روی محور اشتباه نرمال شده باشند، چه معنایی از Attention از دست می‌رود؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-02/31-values.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/31-values.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>